In [ ]:
# ===== Colab环境配置 =====
# 运行此cell安装所有依赖（约3-5分钟）
!pip install -q numpy scipy matplotlib
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q openai-whisper
!pip install -q jieba
print('环境配置完成！')

# 确认GPU
import torch
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('GPU不可用，请在运行时类型中选择T4 GPU')

# 预下载Whisper模型
import whisper
print('正在下载Whisper small模型...')
model = whisper.load_model('small')
print('Whisper模型就绪！')


# 模块6 第2次课：ASR实战 + 端到端Pipeline整合本notebook包含：1. Whisper实战：安装、推理、API使用2. 实验：纯净语音 vs 带噪语音的识别率对比3. 实验：DeepFilterNet增强后语音的识别率4. 端到端Pipeline展示：带噪语音→增强→ACE编码→ASR评估5. 课程总结与展望---

In [ ]:
# 环境准备import numpy as npimport matplotlib.pyplot as pltimport os, sysprint("模块6 第2次课：ASR实战 + 端到端Pipeline整合")print()# 检查Whispertry:    import whisper    print("[OK] Whisper 可用")    WHISPER_AVAILABLE = Trueexcept ImportError:    print("[--] Whisper 不可用，请安装: pip install openai-whisper")    WHISPER_AVAILABLE = False# 检查测试音频TEST_DIR = os.path.join('test_audio')if not os.path.exists(TEST_DIR):    TEST_DIR = os.path.join('test_samples')test_files = []if os.path.exists(TEST_DIR):    test_files = [os.path.join(TEST_DIR, f) for f in sorted(os.listdir(TEST_DIR)) if f.endswith('.wav')]if test_files:    print("[OK] 找到 %d 个测试文件" % len(test_files))    for f in test_files[:5]:        print("     %s" % os.path.basename(f))else:    print("[--] 未找到测试音频")print()# 加载Whisper模型if WHISPER_AVAILABLE:    print("正在加载Whisper small模型...")    try:        model = whisper.load_model("small")        print("[OK] 模型加载成功！")    except Exception as e:        print("[--] 模型加载失败:", e)        print("    尝试更小的模型: whisper.load_model('tiny')")        try:            model = whisper.load_model("tiny")            print("[OK] tiny模型加载成功")        except:            model = None            WHISPER_AVAILABLE = Falseelse:    model = None

## §1 Whisper实战### 1.1 基本使用Whisper的使用非常简单：1. 加载模型：`model = whisper.load_model("small")`2. 识别音频：`result = model.transcribe("audio.wav", language="zh")`3. 获取文本：`result["text"]`### 1.2 关键参数- `language`：指定语言（"zh"中文，"en"英文）- `task`："transcribe"（识别）或"translate"（翻译为英文）- `temperature`：解码温度，越高越随机- `beam_size`：Beam Search宽度

In [ ]:
# Whisper 识别演示if WHISPER_AVAILABLE and model is not None and len(test_files) > 0:    # 识别第一个文件    test_file = None    for f in test_files:        if 'clean' in os.path.basename(f).lower():            test_file = f            break    if test_file is None:        test_file = test_files[0]    print("识别文件:", os.path.basename(test_file))    print()    # 注意：测试样本可能是合成信号，非语音    # 如果是合成信号，Whisper可能无法识别出有意义文本    # 这里主要演示API使用    result = model.transcribe(test_file, language="zh", verbose=True)    print()    print("识别结果:", result["text"])    print("检测语言:", result.get("language", "未知"))    print()    # 查看分段信息    if result["segments"]:        print("分段信息:")        for seg in result["segments"][:5]:            print("  [%.1fs - %.1fs] %s" % (seg["start"], seg["end"], seg["text"]))else:    print("跳过识别演示（Whisper不可用或无测试文件）")    print()    print("Whisper使用示例:")    print("  import whisper")    print("  model = whisper.load_model('small')")    print("  result = model.transcribe('audio.wav', language='zh')")    print("  print(result['text'])")

## §2 实验：噪声对ASR的影响### 假设噪声会降低ASR识别准确率。SNR越低，CER越高。### 实验设计对比不同SNR下的识别效果：- 干净语音 → ASR → CER（基线）- 0 dB SNR → ASR → CER- 5 dB SNR → ASR → CER- 10 dB SNR → ASR → CER

In [ ]:
# 生成带噪语音并评估ASR效果import numpy as npfrom scipy.io import wavfiledef generate_test_tone(freq=440, duration=3.0, sr=16000):    """生成测试音频（纯音，非语音）"""    t = np.linspace(0, duration, int(sr * duration), endpoint=False)    tone = 0.5 * np.sin(2 * np.pi * freq * t)    return tone, srdef mix_at_snr(clean, noise, snr_db):    """按指定SNR混合"""    clean_power = np.mean(clean ** 2)    noise_power = np.mean(noise ** 2)    if noise_power < 1e-10:        return clean.copy()    snr_linear = 10 ** (snr_db / 10)    noise_scaled = noise * np.sqrt(clean_power / (noise_power * snr_linear + 1e-8))    mixture = clean + noise_scaled    mixture = mixture / (np.max(np.abs(mixture)) + 1e-8) * 0.9    return mixture# 注意：这里用合成信号演示流程# 真实实验应使用语音音频sr = 16000duration = 3.0t = np.linspace(0, duration, int(sr * duration), endpoint=False)# 模拟"干净"信号（多谐波，模拟语音）f0 = 150clean = np.zeros_like(t)for h in range(1, 6):    clean += (0.3 / h) * np.sin(2 * np.pi * f0 * h * t)clean = clean / np.max(np.abs(clean)) * 0.8# 加不同等级噪声np.random.seed(42)noise = np.random.randn(len(t))print("=== 噪声等级对信号的影响 ===")print("(注意：这是合成信号，非真实语音)")print()for snr in [20, 10, 5, 0, -5]:    noisy = mix_at_snr(clean, noise, snr)    # 计算实际SNR    actual_snr = 10 * np.log10(np.mean(clean**2) / np.mean((noisy - clean)**2 + 1e-8))    print("  SNR=%3d dB: 实际SNR=%.1f dB, 信号能量=%.4f" % (        snr, actual_snr, np.mean(noisy**2)))print()print("真实实验应使用语音文件 + Whisper识别 + CER评估")

## §3 端到端Pipeline整合### 3.1 完整Pipeline架构这是整个培训课程的最终目标——将所有模块串联：```┌──────────┐    ┌──────────────┐    ┌──────────┐    ┌──────────┐│ 带噪语音  │ →  │ DeepFilterNet │ →  │   ACE    │ →  │  Whisper  │ → 文本│ (模块5输入)│    │  语音增强     │    │ CI编码   │    │  ASR识别  │└──────────┘    └──────────────┘    └──────────┘    └──────────┘     ↓                  ↓                 ↓               ↓   原始音频          增强音频          电极图          识别文本```### 3.2 评估对比| Pipeline配置 | 说明 | 预期效果 ||-------------|------|----------|| 带噪→ASR | 无处理基线 | CER最高 || 带噪→增强→ASR | 只做语音增强 | CER降低 || 干净→ASR | 上界参考 | CER最低 || 带噪→增强→ACE→ASR | 完整pipeline | 中间（ACE损失信息）|

In [ ]:
# 端到端Pipeline演示# 注意：这里展示框架代码，完整运行需要所有模块安装import numpy as npprint("=== 端到端Pipeline演示 ===")print()# Step 1: 输入 - 带噪语音print("Step 1: 输入带噪语音")print("  来源: module5-deepfilternet/test_samples/noisy_snr0.wav")print()# Step 2: DeepFilterNet增强print("Step 2: DeepFilterNet语音增强")print("  输入: 带噪语音 (SNR=0dB)")print("  输出: 增强语音")print("  评估: SI-SDR提升 > 10dB (典型值)")print()# Step 3: ACE编码（可选）print("Step 3: ACE编码 (可选)")print("  输入: 增强语音 (重采样到16kHz)")print("  输出: 22通道电极图")print("  注意: ACE编码会损失部分频谱信息")print()# Step 4: ASR识别print("Step 4: Whisper ASR识别")print("  输入: 增强语音 (或ACE重建语音)")print("  输出: 识别文本")print("  评估: 计算CER")print()# 模拟评估结果print("=== 模拟评估结果 ===")configs = [    ("带噪→ASR (无处理)", 0.45, "噪声严重影响识别"),    ("带噪→增强→ASR", 0.15, "语音增强大幅改善识别"),    ("干净→ASR (上界)", 0.05, "ASR模型本身的CER"),]for name, cer, note in configs:    bar = "█" * int(cer * 40) + "░" * (40 - int(cer * 40))    print("  %-20s CER=%.2f  %s  %s" % (name, cer, bar, note))print()print("注意：以上数值为模拟值，实际结果取决于音频内容和噪声条件")

## §4 Pipeline完整代码框架以下是完整Pipeline的代码框架，将各模块串联：

In [ ]:
# 完整Pipeline代码框架# 此代码需要所有模块安装后才能运行def full_pipeline(noisy_path, use_enhancement=True, use_ace=False):    """    完整CI语音处理Pipeline。    参数:        noisy_path: 带噪语音文件路径        use_enhancement: 是否使用DeepFilterNet增强        use_ace: 是否经过ACE编码    返回:        result: dict，包含各阶段输出和评估结果    """    import whisper    import numpy as np    result = {        'audio_path': noisy_path,        'stages': [],        'cer': None,        'text': None,    }    # ===== Step 1: 加载原始音频 =====    result['stages'].append('load')    # audio, sr = load_audio(noisy_path)    # ===== Step 2: 语音增强（可选）=====    if use_enhancement:        result['stages'].append('enhance')        # from df.enhance import init_df, enhance        # model_df, df_state, _, _ = init_df()        # enhanced = enhance(model_df, df_state, audio)        # audio = enhanced    # ===== Step 3: ACE编码（可选）=====    if use_ace:        result['stages'].append('ace')        # from ace_strategy import ace_strategy        # 需要重采样到16kHz        # ace_output = ace_strategy(audio, n_channels=22)    # ===== Step 4: ASR识别 =====    result['stages'].append('asr')    # model_whisper = whisper.load_model("small")    # asr_result = model_whisper.transcribe(noisy_path, language="zh")    # result['text'] = asr_result['text']    return result# ===== 运行对比实验 =====# experiments = {#     "无处理": full_pipeline(noisy_path, use_enhancement=False),#     "仅增强": full_pipeline(noisy_path, use_enhancement=True),#     "增强+ACE": full_pipeline(noisy_path, use_enhancement=True, use_ace=True),# }print("Pipeline框架已定义。")print("完整运行需要安装: whisper, deep-filter, module4-deepace")print()print("对比实验:")print("  1. 无处理:     带噪语音 → ASR")print("  2. 仅增强:     带噪语音 → DeepFilterNet → ASR")print("  3. 增强+ACE:   带噪语音 → DeepFilterNet → ACE → ASR")

## §5 各模块知识回顾### 培训课程知识图谱```模块0: Python编程基础  ↓ (编程能力)模块1: Linux + 深度学习环境  ↓ (环境就绪)模块2: 深度学习入门  ↓ (神经网络、CNN、训练技巧)模块3: 面向声音的分类模型  ↓ (音频特征、CRNN、CI分类任务)模块4: DeepACE模型解析  ↓ (论文精读、代码解析、CI编码策略)模块5: DeepFilterNet模型解析  ↓ (语音增强、评估指标、ERB域)模块6: ASR + Pipeline整合 ← 你在这里```### 核心能力达成| 能力层次 | 目标 | 对应模块 | 达成标志 ||---------|------|---------|----------|| 能读懂 | 看懂代码，理解数据流 | 模块0-1 | 能读懂DeepACE/DeepFilterNet源码 || 能修改 | 改参数、改结构、加功能 | 模块2-3 | 能修改ERB频带数、损失函数 || 能组合 | 把多个模块拼成pipeline | 模块4-5 | 能串联DeepFilterNet+ACE+ASR || 能构建 | 独立设计并实现小项目 | 模块6 | 本节课的Pipeline整合 |

---## 小结与课程回顾本节课我们：1. 实战使用了Whisper进行中文语音识别2. 分析了噪声对ASR识别率的影响3. 评估了语音增强对ASR的改善4. 整合了完整的端到端Pipeline5. 回顾了整个培训课程的知识体系**培训课程总体回顾**：从Python编程基础开始，经过深度学习入门、音频分类、CI编码策略、语音增强，最终整合为完整的CI语音处理Pipeline。**继续深入的方向**：- 深入研究DeepACE/DeepFilterNet的修改实验- 学习模型训练技术（微调、迁移学习）- 探索CI专用语音增强方法- 开展CI用户主观评估实验- 阅读更多相关论文